# 📊 MESA Benchmark Suite — Zero-Cost Colab Mode

This notebook runs the official **MESA Benchmark System (v0.5.1+)** entirely on Google Colab's free tier, using **Ollama** for zero-cost local LLM Judge evaluations.

It will ingest the synthetic dataset and generate a comprehensive markdown report with Latency, Hit@K, and LLM Judge Accuracy metrics.

---
## 1. Install MESA and Dependencies

In [ ]:
!apt-get update
!apt-get install -y zstd

# Clone the repository
!git clone https://github.com/Yasou13/MESA.git
%cd MESA

# Install Core MESA Dependencies
!pip install -r requirements-core.txt

# Install Benchmark Dependencies
%cd mesa-benchmark
!pip install -r requirements.txt

---
## 2. Install and Start Ollama
We use `llama3.2:3b` as the LLM Judge. We configure the standard OpenAI client to route to Ollama's local OpenAI-compatible endpoint.

In [ ]:
import os
import time

# Route OpenAI python client to local Ollama instance
os.environ["OPENAI_BASE_URL"] = "http://127.0.0.1:11434/v1"
os.environ["OPENAI_API_KEY"] = "ollama"

# Install Ollama binary
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server in the background
!nohup ollama serve > /dev/null 2>&1 &
time.sleep(5)  # Wait for API to boot
print('✅ Ollama server started')

# Pull the model
!ollama pull llama3.2:3b
print('✅ Model ready')

---
## 3. Configure the Benchmark
Update `config.yaml` to explicitly use our local Ollama model for the LLM Judge.

In [ ]:
import yaml

# Update config.yaml to use llama3.2:3b for LLM Judge
with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

config['evaluation']['llm_judge_model'] = 'llama3.2:3b'

with open('config.yaml', 'w') as f:
    yaml.dump(config, f)

print("✅ Updated config.yaml to use Ollama LLM Judge")

---
## 4. Run the MESA Benchmark Suite
This command executes the new hardened benchmark runner.

In [ ]:
# Execute the new benchmark orchestrator
!python -m mesa_benchmark

---
## 5. Visualise the Markdown Report
The benchmark automatically generates a beautiful Markdown report containing all metrics.

In [ ]:
import os
import glob
from IPython.display import Markdown, display

# Find the latest generated markdown report
reports = glob.glob('reports/*.md')
if reports:
    latest_report = max(reports, key=os.path.getctime)
    print(f"Displaying {latest_report}\n")
    with open(latest_report, 'r') as f:
        display(Markdown(f.read()))
else:
    print("⚠️ No report found. Make sure the benchmark completed successfully.")